Notebook for exploring the NZGMDB dataset.

In [ ]:
from pathlib import Path
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sim_ranking as sr

sns.set_theme(style="white")

In [ ]:
# Disable FutureWarning
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# Config
# nzgmdb_ffp = Path("/Users/claudy/dev/work/data/gm_datasets/nz_gmdb/v3.0/Tables/ground_motion_im_table_rotd50_flat.csv")
# nzgmdb_ffp = Path("/Users/claudy/dev/work/data/gm_datasets/nz_gmdb/v3.3/Tables/ground_motion_im_table_rotd50_flat.csv")
# nzgmdb_ffp = Path("/Users/claudy/dev/work/data/gm_datasets/nz_gmdb/v3.4/Tables/ground_motion_im_table_rotd50_flat.csv")
nzgmdb_ffp = Path("/Users/claudy/dev/work/data/gm_datasets/nz_gmdb/v4.0/Tables/ground_motion_im_table_rotd50_flat.csv")

**NZGMDB version:** `{python} str(nzgmdb_ffp.parent.parent.name)`

In [ ]:
obs_data = sr.ObservedData.from_nzgmdb_flat(nzgmdb_ffp, sr.constants.NZGMDBVersion.v4p0)
obs_data.drop_nan()
print(f"Number of records: {obs_data.n_records} after dropping rows with NaN values")

# Apply some sanity filters
obs_data.metadata_filter(dict(rrup=(0, 250), is_ground_level=True))
print(f"Number of records: {obs_data.n_records} after filtering")

# Drop duplicates
obs_data.drop_duplicates(["event_id", "site_id"])
print(f"Number of records: {obs_data.n_records} after dropping duplicates")

record_df = obs_data.record_df
site_df = obs_data.site_df

### Distance and Magnitude Distribution

In [ ]:
# Plot magnitude & distance distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

sns.histplot(record_df["mag"], bins=20, ax=ax1)
ax1.set_title("Magnitude Distribution")
ax1.set_xlabel("Magnitude")
ax1.set_xlim(record_df["mag"].min(), record_df["mag"].max())

sns.histplot(record_df["rrup"], bins=20, ax=ax2)
ax2.set_title("$R_{Rup}$ Distribution")
ax2.set_xlabel("$R_{Rup}$")
ax2.set_xlim(record_df["rrup"].min(), record_df["rrup"].max())

fig.tight_layout()


### Magnitude vs Distance

In [ ]:
# Plot magnitude vs distance
g = sns.jointplot(data=record_df, x="rrup", y="mag", kind="hex", marginal_kws=dict(bins=20), height=8, joint_kws=dict(gridsize=40))

### Vs30 & $T_{Site}$ Distribution

In [ ]:
# Plot Vs30 & T0 distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

sns.histplot(site_df["vs30"], bins=20, ax=ax1)
ax1.set_title("Vs30 Distribution")
ax1.set_xlabel("Vs30")
ax1.set_xlim(site_df["vs30"].min(), site_df["vs30"].max())

sns.histplot(site_df["tsite"], bins=20, ax=ax2)
ax2.set_title("T0 Distribution")
ax2.set_xlabel("tsite")
ax2.set_xlim(site_df["tsite"].min(), site_df["tsite"].max())

fig.tight_layout()

### $Z_{1.0}$ & $Z_{2.5}$ Distribution

In [ ]:
# Plot Z1.0 and Z2.5 distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

sns.histplot(site_df["z1p0"], bins=20, ax=ax1)
ax1.set_title("Z1.0 Distribution")
ax1.set_xlabel("Z1.0")
ax1.set_xlim(site_df["z1p0"].min(), site_df["z1p0"].max())

sns.histplot(site_df["z2p5"], bins=20, ax=ax2)
ax2.set_title("Z2.5 Distribution")
ax2.set_xlabel("Z2.5")
ax2.set_xlim(site_df["z2p5"].min(), site_df["z2p5"].max())

fig.tight_layout()

### $F_{Min}$ Distribution

In [ ]:
# Plot Fmin distribution
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

sns.histplot(record_df["fmin_h1"], bins=20, ax=ax1, log_scale=(True, False))
ax1.set_xlim(0.01, 10)

sns.histplot(record_df["fmin_h2"], bins=20, ax=ax2, log_scale=(True, False))
ax2.set_xlim(0.01, 10)

sns.histplot(record_df["fmin_v"], bins=20, ax=ax3, log_scale=(True, False))
ax3.set_xlim(0.01, 10)

### $F_{Min}$ Distribution (Cumulative)

In [ ]:
# Plot Fmin distribution (cumulative)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

sns.histplot(record_df["fmin_h1"], bins=20, ax=ax1, log_scale=(True, False), cumulative=True)
ax1.set_xlim(0.01, 10)

sns.histplot(record_df["fmin_h2"], bins=20, ax=ax2, log_scale=(True, False), cumulative=True)
ax2.set_xlim(0.01, 10)

### $F_{Min}$ Aggregation

In [ ]:
# Aggregation method
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

sns.histplot(record_df.loc[:, [sr.ObservedData.OtherColEnums.FMIN_H1, sr.ObservedData.OtherColEnums.FMIN_H2]].mean(axis=1), bins=20, ax=ax1, log_scale=(True, False))
ax1.set_title("Mean")
ax1.set_xlim(0.01, 10)

sns.histplot(record_df.loc[:, [sr.ObservedData.OtherColEnums.FMIN_H1, sr.ObservedData.OtherColEnums.FMIN_H2]].min(axis=1), bins=20, ax=ax2, log_scale=(True, False))
ax2.set_title("Min")
ax2.set_xlim(0.01, 10)

sns.histplot(record_df.loc[:, [sr.ObservedData.OtherColEnums.FMIN_H1, sr.ObservedData.OtherColEnums.FMIN_H2]].max(axis=1), bins=20, ax=ax3, log_scale=(True, False))
ax3.set_title("Max")
ax3.set_xlim(0.01, 10)

fig.tight_layout()

### $F_{Min}$ Aggregation (Cumulative)

In [ ]:
# Aggregation method
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

sns.histplot(record_df.loc[:, [sr.ObservedData.OtherColEnums.FMIN_H1, sr.ObservedData.OtherColEnums.FMIN_H2]].mean(axis=1), bins=20, ax=ax1, log_scale=(True, False), cumulative=True)
ax1.set_title("Mean")
ax1.set_xlim(0.01, 10)

sns.histplot(record_df.loc[:, [sr.ObservedData.OtherColEnums.FMIN_H1, sr.ObservedData.OtherColEnums.FMIN_H2]].min(axis=1), bins=20, ax=ax2, log_scale=(True, False), cumulative=True)
ax2.set_title("Min")
ax2.set_xlim(0.01, 10)

sns.histplot(record_df.loc[:, [sr.ObservedData.OtherColEnums.FMIN_H1, sr.ObservedData.OtherColEnums.FMIN_H2]].max(axis=1), bins=20, ax=ax3, log_scale=(True, False), cumulative=True)
ax3.set_title("Max")
ax3.set_xlim(0.01, 10)

fig.tight_layout()

### Number of records per period (Mean)

In [ ]:
# Compute number of records per periods 
mean_fmin_agg = record_df.loc[:, [sr.ObservedData.OtherColEnums.FMIN_H1, sr.ObservedData.OtherColEnums.FMIN_H2]].mean(axis=1)
min_fmin_agg = record_df.loc[:, [sr.ObservedData.OtherColEnums.FMIN_H1, sr.ObservedData.OtherColEnums.FMIN_H2]].min(axis=1)
max_fmin_agg = record_df.loc[:, [sr.ObservedData.OtherColEnums.FMIN_H1, sr.ObservedData.OtherColEnums.FMIN_H2]].max(axis=1)

record_count = {}
for cur_key, cur_period in zip(sr.constants.PSA_KEYS, sr.constants.PERIODS):
	cur_fmin = 1 / cur_period
	record_count[cur_key] = []    
	record_count[cur_key].append(np.count_nonzero(mean_fmin_agg <= cur_fmin))
	record_count[cur_key].append(np.count_nonzero(min_fmin_agg <= cur_fmin))
	record_count[cur_key].append(np.count_nonzero(max_fmin_agg <= cur_fmin))
    
    
record_count_df = pd.DataFrame(record_count, index=["Mean", "Min", "Max"]).T
record_count_df.index = sr.constants.PERIODS

In [ ]:
with sns.axes_style("ticks", {"axes.grid": True, "grid.linestyle": "--", "grid.minor": True}):
	fig, ax = plt.subplots(figsize=(12, 6))
	
	sns.lineplot(data=record_count_df, marker="o", ax=ax)
	ax.set_xscale("log")
	ax.set_title("Number of Records per Period")
	ax.set_xlabel("Period (s)")
	ax.set_ylabel("Number of Records")
	ax.set_xlim(0.01, 10)

### Quality Score Distribution

In [ ]:
# Plot quality score distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

sns.histplot(record_df[sr.ObservedData.OtherColEnums.QUALITY_SCORE_H1], bins=20, ax=ax1)
ax1.set_xlim(0.01, 1)

sns.histplot(record_df[sr.ObservedData.OtherColEnums.QUALITY_SCORE_H1], bins=20, ax=ax2)
ax2.set_xlim(0.0, 1)

fig.tight_layout()
